# New Tokyo PoC Script

## 1 · Scene & Environment Setup

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # Use GPU 0, will use CPU otherwise
import numpy as np
import pandas as pd
import json
import csv as _csv
import poc.setup as poc
import poc.utils as utils
import poc.rt as rt
import poc.topology as tp

path_suffix = "/home/polimi"

# Antennas are Car 1: [1, 2], RSU: [30, 31, 40], Car 2: [5, 6, 7]
transmitters = [30, 31, 5, 6]
receivers = [1, 2, 40, 7]

# ── Ray-tracing scene ─────────────────────────────────────────────────────────
scene = poc.setup_scenario(
    file_name=f"{path_suffix}/tokyo-polimi-dt-jsac/scenarios/ookayama_full_flat/ookayama.xml",
    frequency=60e9,
    bandwidth=1000e6,
    verbose=True,
    time_checker=True
)

poc.setup_coordinate_systems(
    # Using the right corner of the porch of the main building (facing north) as reference point
    ref_sionna_x=162.396, ref_sionna_y=85.782, # Sionna RT
    ref_external_x=-13524.5, ref_external_y=-43817.64 # Tokyo Mobility DT
)

poc.setup_antenna_type(
    transmitters=transmitters,
    receivers=receivers,
    pattern="load_custom", # Use custom pattern defined by CSV files
    elevation_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/elevation_beam_16.csv",
    azimuth_csv=f"{path_suffix}/tokyo-polimi-dt-jsac/panasonic-rsu-pattern/azimuth_beam_16.csv",
    polarization="H", # Use horizontal (define how is mounted in setup_antenna_on_object)
    simulate_perfect_beamforming=True,
    use_look_at_ideal_pointing=False, # If False, use the custom point_toward_peer() function that considers the actual antenna orientation and heading, which is more realistic but may not achieve perfect alignment
    beam_sweeping_angle=60 # [-60, 60]
)

# Peers (Tx)->(Rx) are 30->1, 31->7, 6->40, 5->2
poc.setup_wireless_links(
    wireless_links=[(30, 1), (31, 7), (6, 40), (5, 2)] # (Tx, Rx)
)

poc.setup_rt(
    rt_max_depth=3,
    rt_los=True,
    rt_specular_reflection=True,
    rt_diffuse_reflection=True,
    rt_refraction=False,
    rt_diffraction=False,
    rt_corner_diffraction=False
)

poc.setup_filters(
    transmitters=transmitters,
    receivers=receivers,
    use_kalman_filter=True,
    kalman_process_var=0.3,
    kalman_meas_var=0.8, # Lower = trust measurement more
    kalman_rt_var=3      # Higher = trust RT less
)

struc = poc.startup(port=35944)

## 2 · Object configurator

In [ ]:
# ── Asset paths & TX powers ───────────────────────────────────────────────────
CAR_1_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/coms.ply"
CAR_2_MESH       = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/estima.ply"
RSU_MESH         = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/rsu.ply"
TREE_MESH        = f"{path_suffix}/tokyo-polimi-dt-jsac/obj_meshes/tree.ply"


# Add objects to the scene
poc.add_object(ref_obj_id=1, mesh_path=CAR_1_MESH, position=[193.263, 182.911, 0.758495])
poc.add_object(ref_obj_id=2, mesh_path=CAR_2_MESH, position=[213.124, 191.882, 0.874555])
poc.add_object(ref_obj_id=101, mesh_path=RSU_MESH, position=[195.940, 197.486, 5.0/2])

trees = {}
trees[1] = [151.070, 137.283, 8.64/2]
trees[2] = [167.037, 135.432, 8.64/2]
trees[3] = [153.032, 149.437, 8.64/2]
trees[4] = [168.768, 147.313, 8.64/2]
trees[5] = [170.476, 159.486, 8.64/2]
trees[6] = [185.203, 157.361, 8.64/2]
trees[7] = [186.522, 160.437, 8.64/2]

for i in range(1, len(trees) + 1):
    poc.add_tree(ref_tree_id=i, mesh_path=TREE_MESH, position=trees[i])

# Add antennas to each object
# Car 1 (Antennas: 1, 2)
poc.mount_antenna_on_object(ref_obj_id=1, 
                            ant_id=1, # peer=30 - V2I Client (Rx)
                            displacement=[-0.359, -0.048, 0.837505], orientation=[0, 0, 0], mounted_vertically=True)
poc.mount_antenna_on_object(ref_obj_id=1,  
                            ant_id=2, # peer=5 - V2V Client (Rx)
                            displacement=[0.719, -0.027, 0.319505], orientation=[0, 0, 0], mounted_vertically=False)

# RSU (Antennas: 30, 31, 40)
poc.mount_antenna_on_object(ref_obj_id=101,
                            ant_id=30, # peer=1 - I2V Server (Tx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(220), np.radians(10), 0]), mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=101, 
                            ant_id=31, # peer=7 - I2V Server (Tx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(220), np.radians(10), 0]), mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=101, 
                            ant_id=40, # peer=6 - I2V Client (Rx)
                            displacement=[0, 0, 5.0/2 + 0.5], orientation=np.array([np.radians(-45), np.radians(10), 0]), mounted_vertically=False)

# Car 2 (Antennas: 5, 6, 7)
poc.mount_antenna_on_object(ref_obj_id=2, 
                            ant_id=5, #peer=2 - V2V Server (Tx)
                            displacement=[-1.024, -0.048, 1.071445], orientation=[np.pi, 0, 0], mounted_vertically=True,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=2, 
                            ant_id=6, # peer=40 - V2I Server (Tx)
                            displacement=[-0.431, -0.009, 1.071445], orientation=[np.pi, 0, 0], mounted_vertically=False,
                            tx_power_dbm=16)
poc.mount_antenna_on_object(ref_obj_id=2, 
                            ant_id=7, # peer=31 - V2I Client (Rx)
                            displacement=[-0.34, -0.037, 1.072445], orientation=[0, 0, 0], mounted_vertically=True)

## 3 · Movement tests

### In Sionna RT coordinates

In [ ]:
utils.move_object(ref_obj_id=1, 
                  position=[195.584, 189.436, 0.758495],
                  ref_system="sionna",
                  heading_angle=0, 
                  velocity=5,
                  sionna_structure=struc)

utils.move_object(ref_obj_id=2, 
                  position=[204.080, 188.896, 0.975445], 
                  ref_system="sionna",
                  heading_angle=190, 
                  velocity=2,
                  sionna_structure=struc)

### In Tokyo Mobility DT coordinates

In [ ]:
utils.move_object(ref_obj_id=2, 
                  position=[-13456.458984375, -43786.375, 0.975445],
                  heading_angle=186.96327209472656, 
                  velocity=5,
                  sionna_structure=struc)

utils.move_object(ref_obj_id=1, 
                  position=[-13455.1572265625, -43772.3984375, 0.758495],
                  heading_angle=179.9130859375, 
                  velocity=5,
                  sionna_structure=struc)

In [ ]:
scene.preview(resolution=(1000, 1000), show_orientations=True, show_devices=True)

## 4 · Topology computation test

In [ ]:
topology = tp.evaluate_best_topology(sionna_structure=struc)
topology

## 5 · Showtime!

In [ ]:
import time
import json
import csv

udp_socket = struc["udp_socket"]
verbose = struc["verbose"]

def main():
    while True:
        # Receive data from the socket
        payload, address = udp_socket.recvfrom(1024*10)
        struc["latest_msg_address"] = address
        message = payload.decode()
        
        t_tot = time.time()
        # Parse the JSON message
        data = json.loads(message)

        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            msg_entries = data["data"]
        elif isinstance(data, list):
            msg_entries = data
        else:
            msg_entries = [data]

        # Extract message type (if available)
        type_val = None
        if len(msg_entries) > 0 and isinstance(msg_entries[0], dict):
            type_val = msg_entries[0].get("type")
        
        # Is a configuration message
        if isinstance(type_val, str) and "RT_CONFIGURATION" in type_val.upper():
            rt.manage_online_reconfiguration(msg_entries, struc, is_manual_override=False)
            continue
        
        # Is data --> Handle prediction request
        if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
            data = data["data"]

        if verbose:
            print(" ")
            print("  - - - - - - - - - - - - - - - - -  NEW PREDICTION REQUEST  - - - - - - - - - - - - - - - - -  ")
            print(" ")

        # Extract current measured RSSI to calibrate the Digital Twin                        
        current_1vs2 = data[0]["data"]["RSSI_Vehicle1_2"] if data[0]["data"]["RSSI_Vehicle1_2"] != 0 else None
        current_1vs3 = data[0]["data"]["RSSI_Vehicle1_3"] if data[0]["data"]["RSSI_Vehicle1_3"] != 0 else None
        current_2vs3 = data[0]["data"]["RSSI_Vehicle2_3"] if data[0]["data"]["RSSI_Vehicle2_3"] != 0 else None
        
        if struc["use_kalman_filter"]:
            for key, filt in struc["filters"].items():
                filt.predict()
        

        # Update the locations in the scenario
        future_timestamp = data[1]["clock"]
        predicted_p1 = data[1]["data"]["x1"]
        predicted_p2 = data[1]["data"]["x2"]
        predicted_p3 = data[1]["data"]["x3"]
        t = time.time()

        rt.manage_location_message(f"LOC_UPDATE:veh2,{predicted_p2['x']},{predicted_p2['z']},{0.0},{predicted_p2['yaw']},0,0,0", struc)
        rt.manage_location_message(f"LOC_UPDATE:veh3,{predicted_p1['x']},{predicted_p1['z']},{0.0},{predicted_p1['yaw']},0,0,0", struc) # I know they are inverted!
        rt.manage_location_message(f"LOC_UPDATE:veh1,{predicted_p3['x']},{predicted_p3['z']},{0.0},{predicted_p3['yaw']},0,0,0", struc) # I know they are inverted!
    
        location_update_time = (time.time() - t) * 1000
        if struc["time_checker"]:
            print(f"        Location update took: {location_update_time:.2f} ms")

        # Compute filtered RSSI predictions
        t_c = time.time()
        raw_rssi_1vs2 = {}
        raw_rssi_1vs3 = {}
        raw_rssi_2vs3 = {}        

        if struc["use_kalman_filter"]:
            # Update Kalman filters
            rssi_1vs2 = struc["filters"][("1","2")].update(z_meas=current_1vs2, z_rt=raw_rssi_1vs2)
            rssi_1vs3 = struc["filters"][("1","3")].update(z_meas=current_1vs3, z_rt=raw_rssi_1vs3)
            rssi_2vs3 = struc["filters"][("2","3")].update(z_meas=current_2vs3, z_rt=raw_rssi_2vs3)
        
        if struc["use_adaptive_bias_filter"]:
            rssi_1vs2 = struc["filters"][("1","2")].step(predicted_rt=raw_rssi_1vs2, current_meas=current_1vs2)
            rssi_1vs3 = struc["filters"][("1","3")].step(predicted_rt=raw_rssi_1vs3, current_meas=current_1vs3)
            rssi_2vs3 = struc["filters"][("2","3")].step(predicted_rt=raw_rssi_2vs3, current_meas=current_2vs3)

        rssi_prediction_time = (time.time() - t_c) * 1000
        if struc["time_checker"]:
            print(f"        RT and RSSI extraction took: {rssi_prediction_time:.2f} ms")

        if struc["verbose"]:
            print(f"        Raw predicted RSSI:   1vs2={raw_rssi_1vs2:.2f} dBm, 1vs3={raw_rssi_1vs3:.2f} dBm, 2vs3={raw_rssi_2vs3:.2f} dBm")
            print(f"        Filtered RSSI:        1vs2={rssi_1vs2:.2f} dBm, 1vs3={rssi_1vs3:.2f} dBm, 2vs3={rssi_2vs3:.2f} dBm")

        # Prepare the response
        response = [{}]
        response[0]["clock"] = future_timestamp
        response[0]["data"] = {}
        response[0]["data"]["x1"] = predicted_p1
        response[0]["data"]["x2"] = predicted_p2
        response[0]["data"]["x3"] = predicted_p3
        response[0]["data"]["RSSI_Vehicle1_2"] = rssi_1vs2
        response[0]["data"]["RSSI_Vehicle1_3"] = rssi_1vs3
        response[0]["data"]["RSSI_Vehicle2_3"] = rssi_2vs3
        response[0]["data"]["raw_RSSI_Vehicle1_2"] = raw_rssi_1vs2
        response[0]["data"]["raw_RSSI_Vehicle1_3"] = raw_rssi_1vs3
        response[0]["data"]["raw_RSSI_Vehicle2_3"] = raw_rssi_2vs3

        response = json.dumps(response, default=lambda o: float(o) if isinstance(o, np.float32) else o)

        # Send back the response
        #udp_socket.sendto(response.encode(), ("20.0.0.10", 35944))
        udp_socket.sendto(response.encode(), address) # Use this for local testing script
        total_elapsed_time = (time.time() - t_tot) * 1000

        if verbose:
            print(" ")
            print(f"  - - - - - - - - - - - - - - - - -   Handled in {total_elapsed_time:.2f} ms   - - - - - - - - - - - - - - - - -  ")
            print(" ")
            print(" ")
        
        # Logging
        local_timestamp = time.time()
        with open(struc["log_file"], mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([local_timestamp, data[0]["clock"], future_timestamp,
                             message,
                             struc["sionna_location_db"][1]['x'], struc["sionna_location_db"][1]['y'], struc["sionna_location_db"][1]['angle'],
                             struc["sionna_location_db"][2]['x'], struc["sionna_location_db"][2]['y'], struc["sionna_location_db"][2]['angle'],
                             struc["sionna_location_db"][3]['x'], struc["sionna_location_db"][3]['y'], struc["sionna_location_db"][3]['angle'],
                             raw_rssi_1vs2, raw_rssi_1vs3, raw_rssi_2vs3, 
                             rssi_1vs2, rssi_1vs3, rssi_2vs3,
                             location_update_time, rssi_prediction_time, total_elapsed_time, los_1_2, los_1_3, los_2_3])
        
        '''
        from sionna.rt import render_to_file
        # Frame-by-frame exporter
        global frame_num
        print("Exporting frame ", frame_num)
        
        struc["scene"].render_to_file(camera=my_cam, 
                                         filename=f"/home/rpegurri/Tokyo NDT Integration/PoC Tokyo Science/export/frame_{frame_num}.png", 
                                         show_devices=False)
        frame_num += 1
        '''
        

        if message.startswith("SHUTDOWN_SIONNA"):
            print("Got SHUTDOWN_SIONNA message. Bye!")
            udp_socket.close()
            break

# Entry point
if __name__ == "__main__":
    main()

# Utilities

### Displacement calculator
Note: displacement is computed with cars at their default orientation after being imported (facing Ookayama north)

In [ ]:
car_1 = [193.263, 182.911, 0.758495]
car_2 = [213.124, 191.882, 0.874555]

car_1 = scene.get("obj_1").position
car_2 = scene.get("obj_2").position

car_1 = [car_1[0][0], car_1[1][0], car_1[2][0]]
car_2 = [car_2[0][0], car_2[1][0], car_2[2][0]]
print("Car 1 position:", car_1)
print("Car 2 position:", car_2)

# Car 1
ant_1 = [192.904, 182.863, 1.496 + 0.1]
ant_2 = [193.982, 182.884, 0.978 + 0.1]

# Car 2
ant_5 = [212.100, 191.834, 1.746 + 0.20]
ant_6 = [212.693, 191.873, 1.746 + 0.20]
ant_7 = [212.784, 191.845, 1.747 + 0.20]

displacement_1 = np.array(ant_1) - np.array(car_1)
displacement_2 = np.array(ant_2) - np.array(car_1)
displacement_5 = np.array(ant_5) - np.array(car_2)
displacement_6 = np.array(ant_6) - np.array(car_2)
displacement_7 = np.array(ant_7) - np.array(car_2)

print("Displacement Antenna 1:", displacement_1)
print("Displacement Antenna 2:", displacement_2)
print("Displacement Antenna 5:", displacement_5)
print("Displacement Antenna 6:", displacement_6)
print("Displacement Antenna 7:", displacement_7)